In [1]:
import json

from ortools.linear_solver import pywraplp

In [2]:
with open("../data/animal_with_pickups.json", "r") as f:
    animals = json.load(f)

with open("../data/crop_rollouts.json", "r") as f:
    crops = json.load(f)

with open("../data/handmade_dp_candidates.json", "r") as f:
    candidates = json.load(f)

In [3]:
animal_profiles = {f"{animal}_with_care": animals["animals"][animal]["with_care"] for animal in animals["animals"]}
crop_profiles = {f"{crop}_no_fert": crops["crops"][crop]["no_fert"] for crop in crops["crops"]}

profiles = {**animal_profiles, **crop_profiles}

In [4]:
prices = {}
for animal in animals["animals"]:
    prices[f"{animal}_with_care"] = animals["animals"][animal]["base_price"]
for crop in crops["crops"]:
    prices[f"{crop}_no_fert"] = crops["crops"][crop]["base_price"]

costs = {}
for animal in animals["animals"]:
    costs[f"{animal}_with_care"] = animals["animals"][animal]["animal_cost"]
for crop in crops["crops"]:
    costs[f"{crop}_no_fert"] = crops["crops"][crop]["seed_cost"]

prices, costs

({'GOOSE_with_care': 50,
  'COW_with_care': 160,
  'SHEEP_with_care': 200,
  'WHEAT_no_fert': 25,
  'CARROT_no_fert': 35,
  'MELON_no_fert': 250,
  'TOMATO_no_fert': 60,
  'STRAWBERRY_no_fert': 120},
 {'GOOSE_with_care': 300,
  'COW_with_care': 400,
  'SHEEP_with_care': 500,
  'WHEAT_no_fert': 10,
  'CARROT_no_fert': 20,
  'MELON_no_fert': 80,
  'TOMATO_no_fert': 50,
  'STRAWBERRY_no_fert': 100})

In [5]:
for profile, data in profiles.items():
    print(profile, data.keys())

GOOSE_with_care dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'expected_yield'])
COW_with_care dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'expected_yield'])
SHEEP_with_care dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'expected_yield'])
WHEAT_no_fert dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'tile_free_age', 'expected_yield', 'fertilize_ages'])
CARROT_no_fert dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'tile_free_age', 'expected_yield', 'fertilize_ages'])
MELON_no_fert dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'tile_free_age', 'expected_yield', 'fertilize_ages'])
TOMATO_no_fert dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'tile_free_age', 'expected_yield', 'fertilize_ages'])
STRAWBERRY_no_fert dict_keys(['days', 'harvest_ages', 'yield_per_harvest', 'tile_free_age', 'expected_yield', 'fertilize_ages'])


In [6]:
for profile, data in profiles.items():
    data["timeline_ops_count"] = [0 for _ in range(len(data["days"]))]
    data["timeline_values"] = [0 for _ in range(len(data["days"]))]

    for day, actions in enumerate(data["days"]):
        data["timeline_ops_count"][day] = len(actions["actions"])


        for action in actions["actions"]:
            if action == "HARVEST":
                data["timeline_values"][day] += prices[profile] * data["yield_per_harvest"][0]
            elif action == "FEED":
                data["timeline_values"][day] -= costs["WHEAT_no_fert"]
            elif action in ["BUILD_COOP", "BUILD_PASTURE"] or action == "PLANT":
                data["timeline_values"][day] -= costs[profile]

In [7]:
profiles["WHEAT_no_fert"]["timeline_ops_count"], profiles["WHEAT_no_fert"]["timeline_values"]

([2, 1, 1, 1, 2], [-10, 0, 0, 0, 100])

In [8]:
profiles

{'GOOSE_with_care': {'days': [{'age': 0,
    'actions': ['PICKUP', 'PICKUP', 'BUILD_COOP', 'PLACE', 'FEED', 'CARE']},
   {'age': 1, 'actions': ['PICKUP', 'FEED', 'CARE', 'COLLECT_FERTILIZER']},
   {'age': 2, 'actions': ['PICKUP', 'FEED', 'CARE', 'COLLECT_FERTILIZER']},
   {'age': 3, 'actions': ['PICKUP', 'FEED', 'CARE', 'COLLECT_FERTILIZER']},
   {'age': 4,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 5,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 6,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 7,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 8,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 9,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 10,
    'actions': ['PICKUP', 'FEED', 'CARE', 'HARVEST', 'COLLECT_FERTILIZER']},
   {'age': 11,
 

In [9]:
candidate_values = []
horizon = 30  # calendar days 0..29; animal profiles are 30 ages

for candidate in candidates:
    total = 0
    for profile, start in candidate:
        values = profiles[profile]["timeline_values"]
        total += sum(values[: horizon - start])
    candidate_values.append(total)

len(candidate_values)

108

In [10]:
# Create the mip solver with the SCIP backend.
solver = pywraplp.Solver.CreateSolver("SCIP")

if not solver:
    assert False, "No solver found"

In [11]:
num_candidates = len(candidates)
num_tiles = 25
num_days = 30
start_money = 3000

# x[i, j] is an array of 0-1 variables, which will be 1
# if candidate i is assigned to tile j.
x = {}
for i in range(num_candidates):
    for j in range(num_tiles):
        x[i, j] = solver.IntVar(0, 1, "")

In [12]:
# Each tiles is assigned to at most 1 candidate.
for j in range(num_tiles):
    solver.Add(solver.Sum([x[i, j] for i in range(num_candidates)]) <= 1)

# Each candidate can be assigned to at most 1 tile.
for i in range(num_candidates):
    solver.Add(solver.Sum([x[i, j] for j in range(num_tiles)]) <= 1)
    

In [13]:
for day in range(num_days):
    # Tiles 1-9 can have 15 operations in total per day
    terms = []
    for j in range(9):
        for i in range(num_candidates):
            n_ops = 0
            for profile, start in candidates[i]:
                age = day - start
                ops = profiles[profile]["timeline_ops_count"]
                if 0 <= age < len(ops):
                    n_ops += ops[age]
            terms.append(x[i, j] * n_ops)
    solver.Add(solver.Sum(terms) <= 15)

    # Tiles 10-15 can have 13 operations in total per day
    terms = []
    for j in range(9, 15):
        for i in range(num_candidates):
            n_ops = 0
            for profile, start in candidates[i]:
                age = day - start
                ops = profiles[profile]["timeline_ops_count"]
                if 0 <= age < len(ops):
                    n_ops += ops[age]
            terms.append(x[i, j] * n_ops)
    solver.Add(solver.Sum(terms) <= 13)

    # Tiles 16-21 can have 13 operations in total per day
    terms = []
    for j in range(15, 21):
        for i in range(num_candidates):
            n_ops = 0
            for profile, start in candidates[i]:
                age = day - start
                ops = profiles[profile]["timeline_ops_count"]
                if 0 <= age < len(ops):
                    n_ops += ops[age]
            terms.append(x[i, j] * n_ops)
    solver.Add(solver.Sum(terms) <= 13)

    # Tiles 22-25 can have 11 operations in total per day
    terms = []
    for j in range(21, 25):
        for i in range(num_candidates):
            n_ops = 0
            for profile, start in candidates[i]:
                age = day - start
                ops = profiles[profile]["timeline_ops_count"]
                if 0 <= age < len(ops):
                    n_ops += ops[age]
            terms.append(x[i, j] * n_ops)
    solver.Add(solver.Sum(terms) <= 11)

In [14]:
# Bank: start_money + all flows through today never goes negative
cum_terms = []
for day in range(num_days):
    for j in range(num_tiles):
        for i in range(num_candidates):
            day_money = 0
            for profile, start in candidates[i]:
                age = day - start
                values = profiles[profile]["timeline_values"]
                if 0 <= age < len(values):
                    day_money += values[age]
            if day_money:
                cum_terms.append(x[i, j] * day_money)
    solver.Add(start_money + solver.Sum(cum_terms) >= 0)

In [15]:
# Add objective
objective = solver.Objective()
for i in range(num_candidates):
    for j in range(num_tiles):
        objective.SetCoefficient(x[i, j], candidate_values[i])
objective.SetMaximization()


In [16]:
# Solve the problem
solver.Solve()

pressed CTRL-C 1 times (5 times for forcing termination)


1

In [ ]:
print(solver.Objective().Value())
x[i, j].solution_value()